In [1]:
import os
import cv2 as cv
import numpy as np
import pandas as pd 
import tensorflow as tf
import matplotlib.pyplot as plt

2026-03-18 05:48:50.654383: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773812930.881689      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773812930.941691      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773812931.448960      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773812931.449010      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773812931.449013      55 computation_placer.cc:177] computation placer alr

In [2]:
x=[]
y=[]
base='/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
for i in os.listdir(base):
    sec_base=base+'/'+i
    for j in os.listdir(sec_base):
        img=plt.imread(sec_base+'/'+j)
        img=cv.cvtColor(img,cv.COLOR_BGR2RGB)
        img=cv.resize(img,(128,128))
        x.append(img)
        y.append(i)

In [4]:
X_train=np.array(x)
y_train=np.array(y)
X_train/=255.0

In [5]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_train=le.fit_transform(y_train)

In [6]:
model=tf.keras.models.Sequential()

# CNN
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
model.add(base_model)
model.add(tf.keras.layers.GlobalAveragePooling2D())
# ANN

model.add(tf.keras.layers.Dense(16,activation='relu'))
model.add(tf.keras.layers.Dense(16,activation='relu'))
model.add(tf.keras.layers.Dense(4,activation='softmax'))

I0000 00:00:1773813044.897499      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773813044.900062      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │        20,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,278,820 (8.69 MB)

 Trainable params: 20,836 (81.39 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
@tf.function
def train_step(X, y):
    with tf.GradientTape() as tape:
        pred = model(X, training=True)
        loss = loss_fn(y, pred)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

In [9]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
dataset = dataset.shuffle(X_train.shape[0]).batch(batch_size).prefetch(tf.data.AUTOTUNE) 

epochs = 10
loss_val=[]
for i in range(epochs):
    for X_batch, y_batch in dataset:
        loss = train_step(X_batch, y_batch)
    loss_val.append(loss)
    print("Epoch:", i+1, "Loss:", loss.numpy())

I0000 00:00:1773813062.158483     162 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch: 1 Loss: 0.53832906
Epoch: 2 Loss: 0.25035655
Epoch: 3 Loss: 0.37360784
Epoch: 4 Loss: 0.28817096
Epoch: 5 Loss: 0.28534248
Epoch: 6 Loss: 0.20553224
Epoch: 7 Loss: 0.17616247
Epoch: 8 Loss: 0.09075508
Epoch: 9 Loss: 0.2060715
Epoch: 10 Loss: 0.10248374


In [10]:
accuracy=np.mean(np.argmax(model.predict(X_train),axis=1)==y_train)
print("Training Accuracy:", accuracy*100)

I0000 00:00:1773813097.343530     160 service.cc:152] XLA service 0x7fb5340046c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773813097.343571     160 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773813097.343576     160 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
2026-03-18 05:51:44.391939: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-18 05:51:44.529176: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1773813106.008259     160 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for th

175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step
Training Accuracy: 96.85714285714285


In [11]:
x=[]
y=[]
base='/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
for i in os.listdir(base):
    sec_base=base+'/'+i
    for j in os.listdir(sec_base):
        img=plt.imread(sec_base+'/'+j)
        img=cv.cvtColor(img,cv.COLOR_BGR2RGB)
        img=cv.resize(img,(128,128))
        x.append(img)
        y.append(i)

In [16]:
X_test=np.array(x)/255.0
y_test=np.array(y)

In [17]:
y_test=le.transform(y_test)

In [18]:
accuracy=np.mean(np.argmax(model.predict(X_test),axis=1)==y_test)
print("Accuracy:", accuracy*100)

50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
Accuracy: 88.6875
